# Daily Challenge: Text Analysis of Books Using Word Cloud

Completed Colab-ready notebook for the Lewis Carroll text analysis challenge.

Books:

- Alice's Adventures in Wonderland
- Through the Looking-Glass
- A Tangled Tale

## Local Virtual Environment

The course note asks you to create a virtual environment for NLP work. In Google Colab, the runtime is already isolated, so run the setup cell below. If you work locally, use these commands in the project folder:

```bash
python -m venv .venv-nlp
.venv-nlp\Scripts\activate
pip install -r requirements.txt
python -m spacy download en_core_web_sm
```

## Setup

Run this once before the exercise cells.

In [ ]:
%pip install --quiet requests nltk spacy wordcloud matplotlib pandas scikit-learn scipy==1.12.0 --upgrade

import re
import requests
import nltk
import spacy
import pandas as pd
import matplotlib.pyplot as plt

from collections import Counter
from nltk import pos_tag, ne_chunk
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer
from nltk.tokenize import word_tokenize, sent_tokenize
from spacy.cli import download as spacy_download
from wordcloud import WordCloud
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer

nltk_resources = [
    "punkt",
    "punkt_tab",
    "stopwords",
    "averaged_perceptron_tagger",
    "averaged_perceptron_tagger_eng",
    "maxent_ne_chunker",
    "maxent_ne_chunker_tab",
    "words",
]

for resource in nltk_resources:
    nltk.download(resource, quiet=True)

try:
    nlp = spacy.load("en_core_web_sm", disable=["parser", "ner"])
except OSError:
    spacy_download("en_core_web_sm")
    nlp = spacy.load("en_core_web_sm", disable=["parser", "ner"])

nlp.max_length = 2_000_000
print("Setup complete.")

## Text Preprocessing

### 1. Load the Texts With `requests`

`load_texts()` receives a list of URLs, downloads each book, removes the Project Gutenberg header/footer, removes non-word characters with regular expressions, normalizes whitespace, and returns the cleaned corpus.

The header/footer removal is done before the regex cleaning so the `*** START` and `*** END` markers are still available.

In [ ]:
book_urls = [
    "https://www.gutenberg.org/cache/epub/11/pg11.txt",
    "https://www.gutenberg.org/cache/epub/12/pg12.txt",
    "https://www.gutenberg.org/cache/epub/29042/pg29042.txt",
]

book_titles = [
    "Alice's Adventures in Wonderland",
    "Through the Looking-Glass",
    "A Tangled Tale",
]


def remove_gutenberg_boilerplate(text):
    start_match = re.search(r"\*\*\*\s*START OF THE PROJECT GUTENBERG EBOOK.*?\*\*\*", text, flags=re.IGNORECASE | re.DOTALL)
    end_match = re.search(r"\*\*\*\s*END OF THE PROJECT GUTENBERG EBOOK.*?\*\*\*", text, flags=re.IGNORECASE | re.DOTALL)

    start = start_match.end() if start_match else 0
    end = end_match.start() if end_match else len(text)
    return text[start:end]


def clean_non_words(text):
    text = re.sub(r"[^A-Za-z\s]", " ", text)
    text = re.sub(r"\s+", " ", text)
    return text.strip().lower()


def load_texts(urls):
    corpus = []

    for url in urls:
        response = requests.get(url, timeout=30)
        response.raise_for_status()

        text = response.text
        text = remove_gutenberg_boilerplate(text)
        text = clean_non_words(text)
        corpus.append(text)

    return corpus


corpus = load_texts(book_urls)

print(f"Loaded {len(corpus)} books.")
for title, text in zip(book_titles, corpus):
    print(f"{title}: {len(text):,} characters")

### 2. Print the First 200 Characters

The Project Gutenberg license/header and footer are not relevant to literary text analysis, so the loader removes everything before `*** START` and after `*** END`.

In [ ]:
for title, text in zip(book_titles, corpus):
    print(f"\n{title}")
    print(text[:200])

### 3. Tokenize the Texts and Print the First 150 Tokens

In [ ]:
tokenized_books = [word_tokenize(text) for text in corpus]

for title, tokens in zip(book_titles, tokenized_books):
    print(f"\n{title} - first 150 tokens")
    print(tokens[:150])

### 4. Remove Stopwords With NLTK

The checks below compare counts before and after stopword removal for common stopwords.

In [ ]:
stop_words = set(stopwords.words("english"))
tokens_without_stopwords = [
    [token for token in tokens if token.isalpha() and token not in stop_words]
    for tokens in tokenized_books
]

stopword_checks = ["i", "me", "my", "myself", "we", "our", "ours", "ourselves"]

for title, original_tokens, filtered_tokens in zip(book_titles, tokenized_books, tokens_without_stopwords):
    print(f"\n{title}")
    for word in stopword_checks:
        print(
            f"{word:10} before: {original_tokens.count(word):5} | after: {filtered_tokens.count(word):5}"
        )

### 5. Stem Tokens With `PorterStemmer()`

In [ ]:
stemmer = PorterStemmer()
stemmed_books = [
    [stemmer.stem(token) for token in tokens]
    for tokens in tokens_without_stopwords
]

for title, stemmed_tokens in zip(book_titles, stemmed_books):
    print(f"\n{title} - first 50 stemmed tokens")
    print(stemmed_tokens[:50])

### 6. Lemmatize Tokens With spaCy

Lemmatization returns dictionary-like base forms. For example, `was` becomes `be`, while plural nouns can become singular.

In [ ]:
def lemmatize_texts(texts):
    lemmatized_documents = []

    for doc in nlp.pipe(texts, batch_size=1):
        lemmas = [
            token.lemma_.lower()
            for token in doc
            if token.is_alpha and token.lemma_.lower() not in stop_words
        ]
        lemmatized_documents.append(lemmas)

    return lemmatized_documents


lemmatized_books = lemmatize_texts(corpus)

for title, lemmas in zip(book_titles, lemmatized_books):
    print(f"\n{title} - first 50 lemmatized tokens")
    print(lemmas[:50])

### 7. Stemmed vs Lemmatized Tokens

Stemming is rule-based and often chops words down to shorter stems that are not real dictionary words. For example, words such as `curious`, `curiously`, or `curiosity` may be reduced aggressively.

Lemmatization uses vocabulary and grammatical analysis to return real base forms, such as `children` to `child`, `was` to `be`, or `running` to `run`. This makes lemmatized tokens easier to interpret for analysis and visualization. For the BoW, word cloud, and TF-IDF sections, lemmatized tokens are the best choice because they are cleaner and more readable than stems while still reducing repeated word forms.

### 8. Identify POS Tags With NLTK

Full POS tag lists are stored in `pos_tags_by_book`. The notebook prints the first 50 tags and the 10 most common POS tags per book.

In [ ]:
pos_tags_by_book = [pos_tag(tokens) for tokens in tokens_without_stopwords]

for title, tagged_tokens in zip(book_titles, pos_tags_by_book):
    tag_counts = Counter(tag for _, tag in tagged_tokens)
    print(f"\n{title} - first 50 POS tags")
    print(tagged_tokens[:50])
    print("\nMost common POS tags:")
    print(tag_counts.most_common(10))

### 9. Identify Entities With NLTK

`extract_nltk_entities()` uses NLTK tokenization, POS tagging, and named entity chunking. The full extracted entities are saved in `entities_by_book`, and a compact frequency summary is printed for each book.

In [ ]:
def extract_nltk_entities(text):
    entities = []

    for sentence in sent_tokenize(text):
        sentence_tokens = word_tokenize(sentence)
        sentence_pos = pos_tag(sentence_tokens)
        tree = ne_chunk(sentence_pos)

        for subtree in tree:
            if hasattr(subtree, "label"):
                entity_text = " ".join(token for token, _ in subtree.leaves())
                entities.append((entity_text, subtree.label()))

    return entities


entities_by_book = [extract_nltk_entities(text) for text in corpus]

for title, entities in zip(book_titles, entities_by_book):
    print(f"\n{title}")
    print(f"Total entities found: {len(entities)}")
    print("First 25 entities:")
    print(entities[:25])
    print("\nMost common entities:")
    print(Counter(entities).most_common(15))

## Analysing the Text

### 1. Display a Word Cloud of Each Book

In [ ]:
analysis_documents = [" ".join(lemmas) for lemmas in lemmatized_books]

plt.figure(figsize=(18, 6))

for index, (title, document) in enumerate(zip(book_titles, analysis_documents), start=1):
    wordcloud = WordCloud(
        width=900,
        height=500,
        background_color="white",
        colormap="viridis",
        max_words=120,
        random_state=42,
    ).generate(document)

    plt.subplot(1, 3, index)
    plt.imshow(wordcloud, interpolation="bilinear")
    plt.title(title)
    plt.axis("off")

plt.tight_layout()
plt.show()

### 2. BoW: Five Most Frequent Words Across All Books

The best preprocessing output here is the lemmatized text with stopwords removed. It keeps interpretable word forms while combining related forms such as plurals and verb variations.

In [ ]:
count_vectorizer = CountVectorizer()
bow_matrix = count_vectorizer.fit_transform(analysis_documents)
bow_words = count_vectorizer.get_feature_names_out()

total_word_counts = bow_matrix.sum(axis=0).A1
top_5_indices = total_word_counts.argsort()[::-1][:5]
top_5_bow = [(bow_words[index], int(total_word_counts[index])) for index in top_5_indices]

print("Top 5 most frequent words across all books:")
print(top_5_bow)

### 3. Print the BoW and Identify the Numbers

In the sparse BoW representation:

- The first number is the document number, meaning which book row the value belongs to.
- The second number is the word index, meaning the column for a word in the vocabulary.
- The final number is the count, meaning how many times that word appeared in that document.

In [ ]:
print("BoW sparse matrix preview:")
print(bow_matrix[:3, :20])

print("\nVocabulary examples: index -> word")
for index, word in list(enumerate(bow_words))[:20]:
    print(index, "->", word)

bow_df = pd.DataFrame(
    bow_matrix.toarray(),
    index=book_titles,
    columns=bow_words,
)

display(bow_df.iloc[:, :20])

### 4. Pie Plot of the Five Most Frequent Words

In [ ]:
labels = [f"{word} ({count})" for word, count in top_5_bow]
sizes = [count for _, count in top_5_bow]

plt.figure(figsize=(7, 7))
plt.pie(sizes, labels=labels, autopct="%1.1f%%", startangle=90)
plt.title("Top 5 Most Frequent Words Across All Books")
plt.tight_layout()
plt.show()

### 5. BoW Output Analysis

The most frequent BoW words are partly informative, but many are also expected. In Lewis Carroll's books, character names such as `alice`, common dialogue verbs such as `say`, and repeated story words can dominate the frequency list. These words tell us about the main subject and style, but they are not always the most insightful because frequent words are not necessarily distinctive.

## Solving the Frequency Problem Using TF-IDF

### 1. Create a TF-IDF BoW

Since there are only three documents, `min_df=1` and `max_df=2` keep words that appear in at least one document but remove words that appear in all three books.

In [ ]:
tfidf_vectorizer = TfidfVectorizer(min_df=1, max_df=2)
tfidf_matrix = tfidf_vectorizer.fit_transform(analysis_documents)
tfidf_words = tfidf_vectorizer.get_feature_names_out()

tfidf_df = pd.DataFrame(
    tfidf_matrix.toarray(),
    index=book_titles,
    columns=tfidf_words,
)

display(tfidf_df.iloc[:, :20])

### 2. Pie Plots With the Five Most Relevant TF-IDF Words per Document

In [ ]:
def get_top_tfidf_words_for_document(document_index, top_n=5):
    scores = tfidf_matrix[document_index].toarray().ravel()
    top_indices = scores.argsort()[::-1][:top_n]
    return [(tfidf_words[index], float(scores[index])) for index in top_indices if scores[index] > 0]


plt.figure(figsize=(18, 6))

for index, title in enumerate(book_titles):
    top_words = get_top_tfidf_words_for_document(index, top_n=5)
    labels = [f"{word} ({score:.3f})" for word, score in top_words]
    sizes = [score for _, score in top_words]

    plt.subplot(1, 3, index + 1)
    plt.pie(sizes, labels=labels, autopct="%1.1f%%", startangle=90)
    plt.title(title)

plt.tight_layout()
plt.show()

for index, title in enumerate(book_titles):
    print(f"\n{title}")
    print(get_top_tfidf_words_for_document(index, top_n=5))

### TF-IDF Analysis

TF-IDF is more useful than raw BoW when we want distinctive terms. A frequent word that appears in every book is discounted, while a word that appears often in one book but not in the others receives a higher score. This makes the output more insightful for comparing the books.

Some results may still be character names or repeated story-specific words, but they are more likely to reveal what separates one book from the others. For example, a book-specific place, character, or motif can receive a higher TF-IDF score than a general word used across the whole corpus.